# 06 — Bias Correction and Multi-Model Ensemble Analysis
## Part 4 — Multi-Model Ensemble Processing

This part extends the validated bias-correction framework to the complete multi-model ensemble.

Workflow:

**model discovery → harmonization → model-level correction → validation → ensemble statistics → spread → percentiles → agreement → climate-change signal → NetCDF/CSV outputs**

Each model is corrected independently before ensemble statistics are calculated.


## 1. Scientific Objective

A multi-model ensemble should not be treated as one long time series. Each model is an individual realization with its own systematic bias and structural uncertainty.

Therefore:

**Correct each model independently → validate each model → calculate ensemble statistics across the model dimension.**


## 2. Configuration

The paths and periods below are deliberately easy to edit for the repository.

For the six-model workflow, the code automatically discovers all matching NetCDF files rather than hard-coding six filenames.


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

BASE = Path(".")
INPUT_DIR = BASE / "data"

OUTPUT_DIR = BASE / "outputs" / "bias_correction"
CORRECTED_DIR = OUTPUT_DIR / "corrected_models"
ENSEMBLE_DIR = OUTPUT_DIR / "ensemble"
METRICS_DIR = OUTPUT_DIR / "metrics"
FIGURE_DIR = OUTPUT_DIR / "diagnostics"

for d in [CORRECTED_DIR, ENSEMBLE_DIR, METRICS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

VARIABLE = "tasmax"
SCENARIO = "rcp26"

CAL_START = "1981-01-01"
CAL_END = "2000-12-31"
VAL_START = "2001-01-01"
VAL_END = "2010-12-31"

CORRECTION_METHOD = "additive"

print("Variable:", VARIABLE)
print("Scenario:", SCENARIO)
print("Correction:", CORRECTION_METHOD)


## 3. Discover Model Files

In [ ]:
def discover_model_files(directory, pattern="*.nc"):
    return sorted(Path(directory).glob(pattern))

model_files = discover_model_files(INPUT_DIR, "*.nc")

print(f"Found {len(model_files)} NetCDF files")
for f in model_files:
    print(" -", f.name)


## 4. Identify Model Names

Adapt the parser if your filenames contain a more specific model identifier.

Keeping a stable model name is important because it becomes the `model` coordinate in the ensemble dataset.


In [ ]:
def model_name_from_file(path):
    stem = Path(path).stem
    parts = stem.split("_")

    known = [
        "MPI", "MOHC", "CNRM", "IPSL", "ECMWF",
        "CANESM", "HADGEM", "MIROC", "GFDL"
    ]

    for part in parts:
        if any(token in part.upper() for token in known):
            return part

    return stem

model_names = [model_name_from_file(f) for f in model_files]


## 5. Model Registry

In [ ]:
model_registry = pd.DataFrame({
    "model": model_names,
    "file": [str(f) for f in model_files],
})

model_registry


## 6. Check Duplicate Model Identifiers

In [ ]:
duplicates = pd.Series(model_names).value_counts()
duplicates = duplicates[duplicates > 1]

if len(duplicates):
    print("WARNING: duplicate model identifiers:")
    print(duplicates)
else:
    print("No duplicate model identifiers found.")


## 7. Reference Dataset

Set this to the reference dataset already prepared in the repository.

The reference and model data must have compatible units, calendar handling, and spatial grids before final comparison.


In [ ]:
REFERENCE_FILE = INPUT_DIR / "reference.nc"

# Uncomment after setting the correct path:
#
# reference_ds = xr.open_dataset(REFERENCE_FILE)
# reference_da = reference_ds[VARIABLE]


## 8. Time-Dimension Helper

In [ ]:
def get_time_dim(da):
    for name in ["time", "Time", "date", "datetime"]:
        if name in da.dims:
            return name
    raise ValueError("Time dimension not found.")


## 9. Spatial-Dimension Helper

In [ ]:
def get_spatial_dims(da):
    for pair in [
        ("lat", "lon"),
        ("latitude", "longitude"),
        ("y", "x"),
    ]:
        if all(d in da.dims for d in pair):
            return pair

    return tuple(d for d in da.dims if d.lower() != "time")


## 10. Period Selection

In [ ]:
def select_period(da, start, end):
    return da.sel({get_time_dim(da): slice(start, end)})


## 11. Time Alignment

In [ ]:
def align_time(model, reference):
    mt = get_time_dim(model)
    rt = get_time_dim(reference)

    if mt != rt:
        reference = reference.rename({rt: mt})

    return xr.align(model, reference, join="inner")


## 12. Additive Scaling

In [ ]:
def fit_additive_scaling(model_cal, reference_cal):
    t = get_time_dim(model_cal)
    return (
        reference_cal.mean(dim=t, skipna=True)
        - model_cal.mean(dim=t, skipna=True)
    )

def apply_additive_scaling(model, adjustment):
    return model + adjustment


## 13. Multiplicative Scaling

In [ ]:
def fit_multiplicative_scaling(model_cal, reference_cal, minimum_mean=1e-12):
    t = get_time_dim(model_cal)
    model_mean = model_cal.mean(dim=t, skipna=True)
    ref_mean = reference_cal.mean(dim=t, skipna=True)

    return xr.where(
        np.abs(model_mean) > minimum_mean,
        ref_mean / model_mean,
        np.nan,
    )

def apply_multiplicative_scaling(model, ratio):
    return model * ratio


## 14. Variance Scaling

In [ ]:
def fit_variance_scaling(model_cal, reference_cal, minimum_std=1e-12):
    t = get_time_dim(model_cal)

    model_mean = model_cal.mean(dim=t, skipna=True)
    ref_mean = reference_cal.mean(dim=t, skipna=True)
    model_std = model_cal.std(dim=t, skipna=True)
    ref_std = reference_cal.std(dim=t, skipna=True)

    scale = xr.where(
        model_std > minimum_std,
        ref_std / model_std,
        np.nan,
    )

    return {
        "model_mean": model_mean,
        "reference_mean": ref_mean,
        "scale": scale,
    }

def apply_variance_scaling(model, p):
    return p["reference_mean"] + (
        model - p["model_mean"]
    ) * p["scale"]


## 15. Empirical Quantile Mapping — 1D

In [ ]:
def eqm_1d(model_cal, reference_cal, values, n_quantiles=501):
    model_cal = np.asarray(model_cal, dtype=float)
    reference_cal = np.asarray(reference_cal, dtype=float)
    values = np.asarray(values, dtype=float)

    model_cal = model_cal[np.isfinite(model_cal)]
    reference_cal = reference_cal[np.isfinite(reference_cal)]

    result = np.full_like(values, np.nan)

    if model_cal.size < 2 or reference_cal.size < 2:
        return result

    q = np.linspace(0, 1, n_quantiles)
    model_q = np.quantile(model_cal, q)
    reference_q = np.quantile(reference_cal, q)

    model_q, idx = np.unique(model_q, return_index=True)
    reference_q = reference_q[idx]

    valid = np.isfinite(values)

    result[valid] = np.interp(
        values[valid],
        model_q,
        reference_q,
        left=reference_q[0],
        right=reference_q[-1],
    )

    return result


## 16. EQM — Gridded xarray Wrapper

In [ ]:
def eqm_correct(model_cal, reference_cal, values, n_quantiles=501):
    mt = get_time_dim(model_cal)
    rt = get_time_dim(reference_cal)
    vt = get_time_dim(values)

    if mt != "time":
        model_cal = model_cal.rename({mt: "time"})
    if rt != "time":
        reference_cal = reference_cal.rename({rt: "time"})
    if vt != "time":
        values = values.rename({vt: "time"})

    result = xr.apply_ufunc(
        eqm_1d,
        model_cal,
        reference_cal,
        values,
        input_core_dims=[["time"], ["time"], ["time"]],
        output_core_dims=[["time"]],
        vectorize=True,
        dask="parallelized",
        output_dtypes=[float],
        kwargs={"n_quantiles": n_quantiles},
    )

    if vt != "time":
        result = result.rename({"time": vt})

    return result


## 17. Unified Correction Function

In [ ]:
def apply_correction(model, model_cal, reference_cal, method):
    if method == "additive":
        p = fit_additive_scaling(model_cal, reference_cal)
        return apply_additive_scaling(model, p)

    if method == "multiplicative":
        p = fit_multiplicative_scaling(model_cal, reference_cal)
        return apply_multiplicative_scaling(model, p)

    if method == "variance":
        p = fit_variance_scaling(model_cal, reference_cal)
        return apply_variance_scaling(model, p)

    if method == "eqm":
        return eqm_correct(
            model_cal,
            reference_cal,
            model,
            n_quantiles=501,
        )

    raise ValueError(f"Unknown method: {method}")


## 18. Validation Metrics

In [ ]:
def metric_summary(model_da, reference_da):
    model_da, reference_da = xr.align(
        model_da, reference_da, join="inner"
    )

    t = get_time_dim(model_da)
    error = model_da - reference_da

    bias = error.mean(dim=t, skipna=True)
    mae = np.abs(error).mean(dim=t, skipna=True)
    rmse = np.sqrt((error ** 2).mean(dim=t, skipna=True))

    model_std = model_da.std(dim=t, skipna=True)
    ref_std = reference_da.std(dim=t, skipna=True)

    model_anom = model_da - model_da.mean(dim=t, skipna=True)
    ref_anom = reference_da - reference_da.mean(dim=t, skipna=True)

    correlation = (
        (model_anom * ref_anom).mean(dim=t, skipna=True)
        / (model_anom.std(dim=t, skipna=True)
           * ref_anom.std(dim=t, skipna=True))
    )

    return {
        "bias": bias,
        "mae": mae,
        "rmse": rmse,
        "std_ratio": model_std / ref_std,
        "correlation": correlation,
    }


## 19. Spatially Average a Metric

In [ ]:
def spatial_average(da):
    dims = [
        d for d in da.dims
        if d.lower() in {
            "lat", "latitude", "lon", "longitude", "x", "y"
        }
    ]
    return da.mean(dim=dims, skipna=True) if dims else da

def scalar_metric(da):
    return float(spatial_average(da).mean(skipna=True))


## 20. Process One Model

This function:

1. opens one model;
2. separates calibration and validation periods;
3. fits the correction using calibration data;
4. applies it to validation data;
5. calculates raw and corrected metrics.


In [ ]:
def process_model(model_file, reference_da, variable, method):
    print("Processing:", Path(model_file).name)

    with xr.open_dataset(model_file) as ds:
        model = ds[variable].load()

    model_cal = select_period(model, CAL_START, CAL_END)
    model_val = select_period(model, VAL_START, VAL_END)

    ref_cal = select_period(reference_da, CAL_START, CAL_END)
    ref_val = select_period(reference_da, VAL_START, VAL_END)

    model_cal, ref_cal = align_time(model_cal, ref_cal)
    model_val, ref_val = align_time(model_val, ref_val)

    corrected_val = apply_correction(
        model_val,
        model_cal,
        ref_cal,
        method,
    )

    raw_metrics = metric_summary(model_val, ref_val)
    corrected_metrics = metric_summary(corrected_val, ref_val)

    return {
        "corrected_val": corrected_val,
        "reference_val": ref_val,
        "raw_metrics": raw_metrics,
        "corrected_metrics": corrected_metrics,
    }


## 21. Test One Model First

Do not immediately run all six models.

Run one model first and verify:

- dimensions;
- coordinates;
- units;
- time period;
- corrected values;
- memory usage.


In [ ]:
# Example:
#
# test_result = process_model(
#     model_files[0],
#     reference_da,
#     VARIABLE,
#     CORRECTION_METHOD,
# )
#
# print(test_result["corrected_val"])


## 22. Process the Complete Ensemble

In [ ]:
# Example:
#
# results = {}
#
# for model_file in model_files:
#     name = model_name_from_file(model_file)
#     results[name] = process_model(
#         model_file,
#         reference_da,
#         VARIABLE,
#         CORRECTION_METHOD,
#     )
#
# print("Processed models:", list(results))


## 23. Validation Summary Table

In [ ]:
def build_validation_table(results):
    rows = []

    for model, result in results.items():
        for metric in result["raw_metrics"]:
            rows.append({
                "model": model,
                "metric": metric,
                "raw": scalar_metric(result["raw_metrics"][metric]),
                "corrected": scalar_metric(
                    result["corrected_metrics"][metric]
                ),
            })

    return pd.DataFrame(rows)

# validation_table = build_validation_table(results)
# display(validation_table)


## 24. Improvement Columns

In [ ]:
def add_improvement_columns(table):
    table = table.copy()

    table["absolute_change"] = (
        table["corrected"] - table["raw"]
    )

    table["relative_change_percent"] = (
        100
        * table["absolute_change"]
        / table["raw"].replace(0, np.nan)
    )

    return table

# validation_table = add_improvement_columns(validation_table)


## 25. Model Improvement Rate

In [ ]:
def model_improvement_rate(table, metric, lower_is_better=True):
    subset = table[table["metric"] == metric].copy()

    if lower_is_better:
        improved = subset["corrected"].abs() < subset["raw"].abs()
    else:
        improved = subset["corrected"] > subset["raw"]

    return float(improved.mean())


## 26. Stack Corrected Models

The model dimension is explicitly introduced here.

This is the critical step before ensemble statistics are calculated.


In [ ]:
def stack_models(model_data):
    if not model_data:
        raise ValueError("No model data supplied.")

    arrays = [
        da.expand_dims(model=[name])
        for name, da in model_data.items()
    ]

    return xr.concat(
        arrays,
        dim="model",
        join="exact",
        coords="minimal",
        compat="equals",
    )


## 27. Ensemble Mean

In [ ]:
def ensemble_mean(stack):
    return stack.mean(dim="model", skipna=True)


## 28. Ensemble Median

In [ ]:
def ensemble_median(stack):
    return stack.median(dim="model", skipna=True)


## 29. Inter-Model Standard Deviation

In [ ]:
def ensemble_std(stack):
    return stack.std(dim="model", skipna=True)


## 30. Inter-Model Range

In [ ]:
def ensemble_range(stack):
    return (
        stack.max(dim="model", skipna=True)
        - stack.min(dim="model", skipna=True)
    )


## 31. Ensemble Percentiles

In [ ]:
def ensemble_percentiles(stack):
    qs = [0.10, 0.25, 0.50, 0.75, 0.90]

    result = xr.concat(
        [
            stack.quantile(q, dim="model", skipna=True)
            for q in qs
        ],
        dim="percentile",
    )

    return result.assign_coords(
        percentile=[10, 25, 50, 75, 90]
    )


## 32. Complete Ensemble Products

In [ ]:
def calculate_ensemble_products(stack):
    return {
        "mean": ensemble_mean(stack),
        "median": ensemble_median(stack),
        "std": ensemble_std(stack),
        "range": ensemble_range(stack),
        "percentiles": ensemble_percentiles(stack),
    }


## 33. Build the Corrected Model Stack

Use validation-corrected data for validation ensemble diagnostics.

For future projections, build another stack after applying the same fitted model-specific corrections to the future simulations.


In [ ]:
# Example:
#
# corrected_models = {
#     name: result["corrected_val"]
#     for name, result in results.items()
# }
#
# corrected_stack = stack_models(corrected_models)
# ensemble_products = calculate_ensemble_products(corrected_stack)
#
# print(corrected_stack)


## 34. Ensemble Quality Control

In [ ]:
def ensemble_quality_check(stack):
    print("Number of models:", stack.sizes["model"])
    print("Dimensions:", stack.dims)
    print("Shape:", stack.shape)
    print("Models:", list(stack["model"].values))

    missing = stack.isnull().mean(dim="model")
    print(
        "Maximum fraction missing across models:",
        float(missing.max(skipna=True)),
    )


## 35. Save a DataArray

In [ ]:
def save_dataarray(da, path, variable_name):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    da.to_dataset(name=variable_name).to_netcdf(path)
    print("Saved:", path)


## 36. Save Ensemble Mean and Median

In [ ]:
# Example:
#
# save_dataarray(
#     ensemble_products["mean"],
#     ENSEMBLE_DIR / f"{VARIABLE}_{SCENARIO}_ensemble_mean.nc",
#     VARIABLE,
# )
#
# save_dataarray(
#     ensemble_products["median"],
#     ENSEMBLE_DIR / f"{VARIABLE}_{SCENARIO}_ensemble_median.nc",
#     VARIABLE,
# )


## 37. Save Model Spread

In [ ]:
# Example:
#
# save_dataarray(
#     ensemble_products["std"],
#     ENSEMBLE_DIR / f"{VARIABLE}_{SCENARIO}_model_std.nc",
#     "model_std",
# )
#
# save_dataarray(
#     ensemble_products["range"],
#     ENSEMBLE_DIR / f"{VARIABLE}_{SCENARIO}_model_range.nc",
#     "model_range",
# )


## 38. Save Ensemble Percentiles

In [ ]:
# Example:
#
# ensemble_products["percentiles"].to_netcdf(
#     ENSEMBLE_DIR
#     / f"{VARIABLE}_{SCENARIO}_percentiles.nc"
# )


## 39. Threshold Agreement

The fraction of models exceeding a threshold is:

`number of models above threshold / total valid models`

This is useful for climate-impact thresholds.


In [ ]:
def fraction_above(stack, threshold):
    return (stack > threshold).mean(dim="model")

def fraction_below(stack, threshold):
    return (stack < threshold).mean(dim="model")


## 40. Model Climate-Change Signal

Calculate future minus historical mean **for every model first**.

Then summarize those model-level changes.

This preserves information about inter-model differences.


In [ ]:
def model_change_signal(historical_stack, future_stack):
    ht = get_time_dim(historical_stack)
    ft = get_time_dim(future_stack)

    historical_mean = historical_stack.mean(
        dim=ht, skipna=True
    )
    future_mean = future_stack.mean(
        dim=ft, skipna=True
    )

    return future_mean - historical_mean


## 41. Change-Signal Ensemble Statistics

In [ ]:
def summarize_change(change_stack):
    return {
        "mean": change_stack.mean(dim="model", skipna=True),
        "median": change_stack.median(dim="model", skipna=True),
        "std": change_stack.std(dim="model", skipna=True),
        "p10": change_stack.quantile(
            0.10, dim="model", skipna=True
        ),
        "p90": change_stack.quantile(
            0.90, dim="model", skipna=True
        ),
    }


## 42. Direction-of-Change Agreement

In [ ]:
def direction_agreement(change_stack):
    positive = (change_stack > 0).sum(dim="model")
    negative = (change_stack < 0).sum(dim="model")
    total = change_stack.notnull().sum(dim="model")

    return xr.Dataset({
        "positive_fraction": positive / total,
        "negative_fraction": negative / total,
    })


## 43. Spatial Mean Time Series

For a first-order ensemble time series, calculate a spatial mean and then summarize models.

For publication work, consider area weighting, especially over large latitude ranges.


In [ ]:
def spatial_mean(da):
    dims = get_spatial_dims(da)
    return da.mean(dim=list(dims), skipna=True)

def plot_model_timeseries(stack, title):
    ts = spatial_mean(stack)

    plt.figure(figsize=(12, 6))

    for model in ts["model"].values:
        ts.sel(model=model).plot(
            linewidth=1,
            label=str(model),
        )

    ts.mean(dim="model", skipna=True).plot(
        linewidth=2,
        label="Ensemble mean",
    )

    plt.title(title)
    plt.ylabel(VARIABLE)
    plt.legend(
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
    )
    plt.tight_layout()
    plt.show()


## 44. Ensemble Mean with 10–90% Range

In [ ]:
def plot_ensemble_range(stack, title):
    ts = spatial_mean(stack)

    mean_ts = ts.mean(dim="model", skipna=True)
    p10 = ts.quantile(0.10, dim="model", skipna=True)
    p90 = ts.quantile(0.90, dim="model", skipna=True)

    plt.figure(figsize=(12, 6))

    mean_ts.plot(label="Ensemble mean")

    plt.fill_between(
        mean_ts["time"].values,
        p10.values,
        p90.values,
        alpha=0.25,
        label="10–90% range",
    )

    plt.title(title)
    plt.ylabel(VARIABLE)
    plt.legend()
    plt.tight_layout()
    plt.show()


## 45. Spatial Field Plot Helper

In [ ]:
def plot_spatial_field(da, title):
    dims = get_spatial_dims(da)

    if len(dims) != 2:
        raise ValueError("Expected a two-dimensional spatial field.")

    plt.figure(figsize=(10, 6))
    da.plot()
    plt.title(title)
    plt.tight_layout()
    plt.show()


## 46. Plot Inter-Model Spread

In [ ]:
# Example:
#
# spread = ensemble_products["std"]
#
# if "time" in spread.dims:
#     spread = spread.mean(dim="time", skipna=True)
#
# plot_spatial_field(
#     spread,
#     "Inter-model standard deviation",
# )


## 47. Plot Agreement

In [ ]:
# Example:
#
# agreement = direction_agreement(change_stack)
#
# plot_spatial_field(
#     agreement["positive_fraction"],
#     "Fraction of models with positive change",
# )


## 48. Save Validation Metrics

In [ ]:
def save_validation_metrics(table):
    path = METRICS_DIR / (
        f"{VARIABLE}_{SCENARIO}_validation_metrics.csv"
    )
    table.to_csv(path, index=False)
    print("Saved:", path)

# Example:
# save_validation_metrics(validation_table)


## 49. Ensemble Dataset

Store the major ensemble statistics together where practical.

Percentiles are kept separately because they use a `percentile` dimension.


In [ ]:
def build_ensemble_dataset(products):
    return xr.Dataset({
        "ensemble_mean": products["mean"],
        "ensemble_median": products["median"],
        "model_std": products["std"],
        "model_range": products["range"],
    })

# Example:
# ensemble_ds = build_ensemble_dataset(ensemble_products)


## 50. Add Ensemble Metadata

In [ ]:
def add_ensemble_metadata(ds, variable, scenario, method, model_count):
    ds = ds.copy()

    ds.attrs["variable"] = variable
    ds.attrs["scenario"] = scenario
    ds.attrs["bias_correction_method"] = method
    ds.attrs["number_of_models"] = int(model_count)
    ds.attrs["ensemble_processing"] = "multi-model statistics"

    return ds


## 51. Save Complete Ensemble Dataset

In [ ]:
# Example:
#
# ensemble_ds = add_ensemble_metadata(
#     ensemble_ds,
#     VARIABLE,
#     SCENARIO,
#     CORRECTION_METHOD,
#     corrected_stack.sizes["model"],
# )
#
# ensemble_ds.to_netcdf(
#     ENSEMBLE_DIR
#     / f"{VARIABLE}_{SCENARIO}_ensemble.nc"
# )


## 52. Save Model Names

Record the exact model membership of every ensemble product. Ensemble membership should never be ambiguous.


In [ ]:
def save_model_membership(model_names):
    path = METRICS_DIR / (
        f"{VARIABLE}_{SCENARIO}_ensemble_model_membership.csv"
    )

    pd.DataFrame({"model": model_names}).to_csv(
        path, index=False
    )

    print("Saved:", path)

# Example:
# save_model_membership(list(corrected_stack["model"].values))


## 53. Compare Ensemble Mean and Reference

For validation, compare the ensemble mean against the reference in exactly the same way as an individual model.


In [ ]:
# Example:
#
# ensemble_mean_val = ensemble_products["mean"]
# reference_val = next(iter(results.values()))["reference_val"]
#
# ensemble_metrics = metric_summary(
#     ensemble_mean_val,
#     reference_val,
# )
#
# print({
#     name: scalar_metric(value)
#     for name, value in ensemble_metrics.items()
# })


## 54. Ensemble vs Individual Models

Do not assume that the ensemble mean is automatically superior.

Compare:

- each individual model;
- ensemble mean;
- ensemble median;
- reference.

The ensemble can reduce random/internal variability while still retaining systematic structural uncertainty.


## 55. Weighted Ensembles — Concept

A later extension can assign weights using independent validation performance.

For example:

`weight_i ∝ 1 / error_i`

However, weighting must be justified scientifically. Avoid selecting weights using future observations.


In [ ]:
def normalize_weights(weights):
    weights = pd.Series(weights, dtype=float)
    weights = weights.clip(lower=0)

    if weights.sum() == 0:
        raise ValueError("All ensemble weights are zero.")

    return weights / weights.sum()


## 56. Weighted Ensemble Mean

In [ ]:
def weighted_ensemble_mean(stack, weights):
    weights = normalize_weights(weights)

    missing = set(stack["model"].values) - set(weights.index)

    if missing:
        raise ValueError(
            f"Missing weights for models: {sorted(missing)}"
        )

    weight_da = xr.DataArray(
        weights.loc[list(stack["model"].values)].values,
        dims=["model"],
        coords={"model": stack["model"].values},
    )

    return (stack * weight_da).sum(
        dim="model",
        skipna=True,
    )


## 57. Why Equal Weighting Is a Good Baseline

Equal weighting is transparent and reproducible:

`w_i = 1 / N`

It should normally be reported before experimenting with performance-based weights.

Weighted ensembles can be presented as a sensitivity analysis rather than silently replacing the equal-weight ensemble.


In [ ]:
def equal_weights(stack):
    n = stack.sizes["model"]
    return pd.Series(
        1.0 / n,
        index=[str(x) for x in stack["model"].values],
    )

# Example:
# weights = equal_weights(corrected_stack)
# weighted_mean = weighted_ensemble_mean(
#     corrected_stack,
#     weights,
# )


## 58. Physical Plausibility Checks

In [ ]:
def physical_range_check(da, variable):
    print("Variable:", variable)
    print("Minimum:", float(da.min(skipna=True)))
    print("Maximum:", float(da.max(skipna=True)))
    print("NaN count:", int(da.isnull().sum()))

    if variable.lower() in {
        "pr", "precip", "precipitation"
    }:
        negative = int((da < 0).sum())
        print("Negative values:", negative)


## 59. Important Interpretation Rule

Ensemble spread is **not** the same thing as a formal probability distribution.

A 10–90% model range means that 80% of the participating model values lie between those model quantiles. It should not automatically be interpreted as an 80% probability interval for the real climate.


## 60. Recommended Final Ensemble Outputs

For each variable and scenario, retain:

```text
corrected_models/
    <model>_corrected.nc

ensemble/
    <variable>_<scenario>_ensemble.nc
    <variable>_<scenario>_percentiles.nc
    <variable>_<scenario>_ensemble_mean.nc
    <variable>_<scenario>_ensemble_median.nc
    <variable>_<scenario>_model_std.nc
    <variable>_<scenario>_model_range.nc

metrics/
    <variable>_<scenario>_validation_metrics.csv
    <variable>_<scenario>_ensemble_model_membership.csv

diagnostics/
    validation plots
    ensemble time series
    spread maps
    agreement maps
```


## 61. Production Workflow

The complete production sequence is:

1. discover all model files;
2. verify model identifiers;
3. load and validate the reference;
4. harmonize time/calendar/grid/units;
5. define calibration and validation periods;
6. fit a correction independently for each model;
7. validate every corrected model;
8. save individual corrected models;
9. correct future simulations using the historical fit;
10. stack corrected models;
11. calculate ensemble mean and median;
12. calculate model spread and percentiles;
13. calculate climate-change signals;
14. calculate model agreement;
15. export metrics and metadata.


## 62. Final Quality-Control Checklist

Before using the ensemble products in climate-impact or attribution analysis:

- [ ] no duplicate model identifiers;
- [ ] all model units verified;
- [ ] reference units verified;
- [ ] calendars verified;
- [ ] grids verified;
- [ ] calibration period fixed;
- [ ] validation period independent;
- [ ] correction fitted separately for each model;
- [ ] future data never used to fit historical correction;
- [ ] raw data preserved;
- [ ] corrected model files saved;
- [ ] ensemble membership recorded;
- [ ] mean and median calculated;
- [ ] inter-model spread calculated;
- [ ] percentiles calculated;
- [ ] agreement calculated;
- [ ] physical range checks passed;
- [ ] metadata written.


# Part 4 Summary

Part 4 converts the single-model bias-correction framework into a reproducible **multi-model ensemble analysis**.

The main scientific outputs are:

- model-by-model corrected simulations;
- ensemble mean;
- ensemble median;
- inter-model standard deviation;
- inter-model range;
- 10th–90th percentile distribution;
- threshold agreement;
- direction-of-change agreement;
- ensemble climate-change signal;
- optional weighted ensemble sensitivity analysis;
- validation metrics and model membership records.

### Next: Part 5

Part 5 can complete Notebook 06 with **uncertainty decomposition, advanced ensemble weighting, seasonal/extreme-event diagnostics, publication-quality figures, and an automated end-to-end production pipeline**.
